In [1]:
import json

from pathlib import Path

NOTEBOOK_DIR = Path.cwd()                 # .../multiwoz-grice-agreement/notebooks
PROJECT_ROOT = NOTEBOOK_DIR.parent        # .../multiwoz-grice-agreement

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "data.json"

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH:", DATA_PATH.resolve())

assert DATA_PATH.exists(), f"Missing: {DATA_PATH.resolve()}"

with DATA_PATH.open(encoding="utf-8") as f:
    data = json.load(f)

print("Dialogs:", len(data))


NOTEBOOK_DIR: c:\Users\benwe\Documents\GitHub\multiwoz-grice-agreement\notebooks
PROJECT_ROOT: c:\Users\benwe\Documents\GitHub\multiwoz-grice-agreement
DATA_PATH: C:\Users\benwe\Documents\GitHub\multiwoz-grice-agreement\data\raw\data.json
Dialogs: 10438


In [2]:
from ollama import Client
gemma = Client(host="http://hcai.uni-regensburg.de:11434", timeout=500)
gemma.generate(model='gemma3:27b')

llama_3_3 = Client(host="http://intern.schlaubox.de:11434", timeout=500)
llama_3_3.generate(model='llama3:70b')
#llama_3_3.generate(model='deepseek-r1:32b')

GenerateResponse(model='llama3:70b', created_at='2026-01-29T09:11:05.478863439Z', done=True, done_reason='load', total_duration=None, load_duration=None, prompt_eval_count=None, prompt_eval_duration=None, eval_count=None, eval_duration=None, response='', thinking=None, context=None)

In [10]:
from string import Template

SYS_PROMPT = r"""
You are a dialogue COHERENCE judge for task-oriented dialogues.

Your task is to evaluate discourse coherence, not task success.
The dialogues are human-human, scripted interactions created for a dataset.

Judge utterances using a task-specific interpretation of Grice's Cooperative Principle:

- Quality: The utterance does not contradict prior dialogue and handles information consistently.
- Quantity: The utterance provides an appropriate amount of information for the interaction.
- Relation: The utterance is relevant to the preceding dialogue context.
- Manner: The utterance is clear, concise, and unambiguous as a conversational move.

Do NOT evaluate real-world correctness.
Do NOT assume access to external knowledge.
All utterances are to be treated as text within a dialogue, not as factual claims.
"""
GRICE_ROLE_AWARE_PROMPT = Template(r"""
Task:
Score the NEXT turn only.
Judge whether the next turn shows cooperative, role-appropriate behaviour
in a task-oriented dialogue between a CUSTOMER and an EXPERT.

Definition:
Cooperative behaviour concerns how clearly and appropriately a speaker
signals relevance, intent, and alignment with the ongoing interaction.
This judgment is grounded in Grice's Cooperative Principle,
interpreted at the discourse level (not task success).

Role expectations:

CUSTOMER (cooperative participation):
- Expresses an interpretable goal or response (Quality)
- Provides relevant information when appropriate (Relation)
- Is consistent or clearly signals corrections (Quantity)
- May be vague unless vagueness blocks progress (Manner)

EXPERT (assistance behaviour; coherence-scoped):
- Handles information consistently without contradiction (Quality)
- Responds to CUSTOMER intent at the discourse level (Relation)
- Provides as much information as needed for the interaction (Quantity)
- Is clear enough to function as a conversational move (Manner)
                                  
Scoring:

Neutral behaviour:
Turns that are interactionally appropriate but task-inert
(e.g., greetings, acknowledgments, closings).
Neutral turns are NOT failures unless they obstruct or mislead.
                                   
Labels:
Very poor | Poor | Weak | Neutral | Good | Very good | Excellent

Notes:
- For CUSTOMER turns, scores reflect cooperative participation, not assistance quality.
- Do not over-reward politeness or verbosity.
- Reserve “Excellent” for contextually outstanding behaviour.

Instructions:
- Score ONLY the next turn.
- If the turn contains multiple information units, split them.
- Split only at sentence boundaries or clearly separate requests/claims.
- Reduce each unit to its task-relevant core.
- Copy the exact PHRASE from the next turn.
- "related_text": MUST copy the most relevant earlier phrase. Use "" only if the next turn is completely unrelated to the dialogue so far.
- Prefer the most recent relevant one.
- Keep reasoning to MAXIMUM 5 words.

Return ONLY a JSON array. No surrounding text.

Each array item must be:
{
  "text": "<phrase from next turn>",
  "related_text": "<phrase from earlier, related unit or ''>",
  "reasoning": "<≤5 words>",
  "score": "<one label>"
}

Dialogue so far:
$dialogue

Next turn:
$last_utterance
""".strip())

In [4]:
TEST_DIALOG_IDS = {
    "PMUL3860.json",
}

selected_dialogs = {k: v for k, v in data.items() if k in TEST_DIALOG_IDS}

missing = TEST_DIALOG_IDS - selected_dialogs.keys()
assert not missing, f"Missing dialogs: {missing}"

print("Selected dialogs:", list(selected_dialogs.keys()))


Selected dialogs: ['PMUL3860.json']


In [5]:
print("Selected dialogs:", len(selected_dialogs))
print("IDs:", list(selected_dialogs.keys()))

missing = TEST_DIALOG_IDS - selected_dialogs.keys()
assert not missing, f"Missing dialogs: {missing}"


Selected dialogs: 1
IDs: ['PMUL3860.json']


In [6]:
from pathlib import Path
import json
import random
from datetime import datetime

# -----------------------
# Config
# -----------------------
SEED = 42
SAMPLE_K = 6

OUT_DIR = Path("sanity_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_JSONL = OUT_DIR / f"grice_sanity_{SAMPLE_K}_seed{SEED}_{ts}.jsonl"
OUT_MD    = OUT_DIR / f"grice_sanity_{SAMPLE_K}_seed{SEED}_{ts}.md"

rng = random.Random(SEED)

# Keep dialog_id!
dialog_ids = list(data.keys())
sample_ids = rng.sample(dialog_ids, k=min(SAMPLE_K, len(dialog_ids)))

print("Sample dialog_ids:")
for d in sample_ids:
    print(" -", d)

def _safe_json_loads(s: str):
    """Parse model output; return (ok, parsed_or_error_str)."""
    try:
        parsed = json.loads(s)
        if not isinstance(parsed, list):
            return False, "Model output is not a JSON array (expected list)."
        return True, parsed
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"

def _turn_speaker(idx: int) -> str:
    """MultiWOZ logs alternate: user first, then system."""
    return "customer" if (idx % 2 == 0) else "expert"

# -----------------------
# Run sanity sample
# -----------------------
all_grice_units = []
md_lines = []
md_lines.append(f"# Grice sanity sample\n")
md_lines.append(f"- seed: {SEED}\n- sample_k: {SAMPLE_K}\n- jsonl: {OUT_JSONL.as_posix()}\n\n")

with OUT_JSONL.open("w", encoding="utf-8") as f_jsonl:
    for n, dialog_id in enumerate(sample_ids, start=1):
        dialog = data[dialog_id]
        log = dialog.get("log", [])

        md_lines.append(f"\n## {n}. {dialog_id}\n")

        dialogue_so_far = ""  # what you feed as $dialogue
        for idx, turn in enumerate(log):
            text = (turn.get("text") or "").strip()
            speaker = _turn_speaker(idx)
            last_utterance = f"{speaker}: {text}\n"

            # Human-readable transcript in MD
            md_lines.append(f"**Turn {idx} ({speaker})**: {text}\n")

            messages = [
                {"role": "system", "content": SYS_PROMPT},
                {"role": "user", "content": GRICE_ROLE_AWARE_PROMPT.substitute(
                    dialogue=dialogue_so_far,
                    last_utterance=last_utterance,
                )},
            ]

            # Call model
            mess = llama_3_3.chat(model="llama3:70b", messages=messages)
            llm_output = mess["message"]["content"]

            ok, parsed_or_err = _safe_json_loads(llm_output)

            if not ok:
                # Write error record to JSONL
                err_record = {
                    "_dialog_id": dialog_id,
                    "_turn_index": idx,
                    "_speaker": speaker,
                    "_source_text": text,
                    "_error": parsed_or_err,
                    "_raw_output_first400": llm_output[:400],
                }
                f_jsonl.write(json.dumps(err_record, ensure_ascii=False) + "\n")

                md_lines.append(f"> ⚠️ **Parse error**: {parsed_or_err}\n")
                md_lines.append(f"> Raw (first 200 chars): `{llm_output[:200].replace('`','\\`')}`\n\n")

            else:
                parsed = parsed_or_err

                # Pretty-print units in MD
                md_lines.append(f"\n<details><summary>Units ({len(parsed)})</summary>\n\n```json\n")
                md_lines.append(json.dumps(parsed, ensure_ascii=False, indent=2))
                md_lines.append("\n```\n</details>\n\n")

                # Write each unit as one JSONL row
                for unit_i, unit in enumerate(parsed):
                    if not isinstance(unit, dict):
                        continue

                    unit_record = {
                        **unit,
                        "_dialog_id": dialog_id,
                        "_turn_index": idx,
                        "_unit_index": unit_i,
                        "_speaker": speaker,
                        "_source_text": text,
                    }
                    all_grice_units.append(unit_record)
                    f_jsonl.write(json.dumps(unit_record, ensure_ascii=False) + "\n")

            # Update running dialogue context
            dialogue_so_far += last_utterance

        print(f"[{n}/{len(sample_ids)}] done:", dialog_id)

# Write MD report
OUT_MD.write_text("".join(md_lines), encoding="utf-8")

print("\nWrote:")
print(" -", OUT_JSONL.resolve())
print(" -", OUT_MD.resolve())
print("Units captured:", len(all_grice_units))


Sample dialog_ids:
 - PMUL3276.json
 - MUL2453.json
 - SNG0234.json
 - PMUL4570.json
 - PMUL1915.json
 - PMUL3860.json


KeyboardInterrupt: 

In [11]:
from pathlib import Path
import json
from datetime import datetime

# -----------------------
# Fixed-dialog sanity run (ready to run)
# Assumes `data`, `SYS_PROMPT`, `GRICE_ROLE_AWARE_PROMPT`, and `llama_3_3` already exist.
# -----------------------

# Your fixed dialog IDs (MultiWOZ keys include ".json")
TEST_DIALOG_IDS = [
    "PMUL3860.json",
]

OUT_DIR = Path("sanity_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TAG = "fixed_debug"
OUT_JSONL = OUT_DIR / f"{RUN_TAG}_grice_sanity_{ts}.jsonl"
OUT_MD    = OUT_DIR / f"{RUN_TAG}_grice_sanity_{ts}.md"

def _safe_json_loads(s: str):
    """Parse model output; return (ok, parsed_or_error_str)."""
    try:
        parsed = json.loads(s)
        if not isinstance(parsed, list):
            return False, "Model output is not a JSON array (expected list)."
        return True, parsed
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"

def _turn_speaker(idx: int) -> str:
    """MultiWOZ logs alternate: user first, then system."""
    return "customer" if (idx % 2 == 0) else "expert"

# -----------------------
# Select fixed dialogs (and warn if any are missing)
# -----------------------
sample_ids = [d for d in TEST_DIALOG_IDS if d in data]
missing = [d for d in TEST_DIALOG_IDS if d not in data]

print("Selected dialog_ids:")
for d in sample_ids:
    print(" -", d)

if missing:
    print("\n⚠️ Missing (not in this data.json):")
    for d in missing:
        print(" -", d)

assert sample_ids, "None of the requested dialog IDs were found in `data`."

# -----------------------
# Run fixed sanity sample
# -----------------------
all_grice_units = []
md_lines = []
md_lines.append("# Grice sanity sample (fixed dialogs)\n\n")
md_lines.append(f"- requested: {len(TEST_DIALOG_IDS)}\n")
md_lines.append(f"- found: {len(sample_ids)}\n")
md_lines.append(f"- jsonl: {OUT_JSONL.as_posix()}\n\n")

with OUT_JSONL.open("w", encoding="utf-8") as f_jsonl:
    for n, dialog_id in enumerate(sample_ids, start=1):
        dialog = data[dialog_id]
        log = dialog.get("log", [])

        md_lines.append(f"\n## {n}. {dialog_id}\n")

        dialogue_so_far = ""  # what you feed as $dialogue
        for idx, turn in enumerate(log):
            text = (turn.get("text") or "").strip()
            speaker = _turn_speaker(idx)
            last_utterance = f"{speaker}: {text}\n"

            # Human-readable transcript in MD
            md_lines.append(f"**Turn {idx} ({speaker})**: {text}\n")

            messages = [
                {"role": "system", "content": SYS_PROMPT},
                {"role": "user", "content": GRICE_ROLE_AWARE_PROMPT.substitute(
                    dialogue=dialogue_so_far,
                    last_utterance=last_utterance,
                )},
            ]

            # Call model
            mess = llama_3_3.chat(model="llama3:70b", messages=messages)
            llm_output = mess["message"]["content"]

            ok, parsed_or_err = _safe_json_loads(llm_output)

            if not ok:
                # Write error record to JSONL
                err_record = {
                    "_dialog_id": dialog_id,
                    "_turn_index": idx,
                    "_speaker": speaker,
                    "_source_text": text,
                    "_error": parsed_or_err,
                    "_raw_output_first400": llm_output[:400],
                }
                f_jsonl.write(json.dumps(err_record, ensure_ascii=False) + "\n")

                md_lines.append(f"> ⚠️ **Parse error**: {parsed_or_err}\n")
                md_lines.append(f"> Raw (first 200 chars): `{llm_output[:200].replace('`','\\`')}`\n\n")

            else:
                parsed = parsed_or_err

                # Pretty-print units in MD
                md_lines.append(f"\n<details><summary>Units ({len(parsed)})</summary>\n\n```json\n")
                md_lines.append(json.dumps(parsed, ensure_ascii=False, indent=2))
                md_lines.append("\n```\n</details>\n\n")

                # Write each unit as one JSONL row
                for unit_i, unit in enumerate(parsed):
                    if not isinstance(unit, dict):
                        continue

                    unit_record = {
                        **unit,
                        "_dialog_id": dialog_id,
                        "_turn_index": idx,
                        "_unit_index": unit_i,
                        "_speaker": speaker,
                        "_source_text": text,
                    }
                    all_grice_units.append(unit_record)
                    f_jsonl.write(json.dumps(unit_record, ensure_ascii=False) + "\n")

            # Update running dialogue context
            dialogue_so_far += last_utterance

        print(f"[{n}/{len(sample_ids)}] done:", dialog_id)

# Write MD report
OUT_MD.write_text("".join(md_lines), encoding="utf-8")

print("\nWrote:")
print(" -", OUT_JSONL.resolve())
print(" -", OUT_MD.resolve())
print("Units captured:", len(all_grice_units))


Selected dialog_ids:
 - PMUL3860.json
[1/1] done: PMUL3860.json

Wrote:
 - C:\Users\benwe\Documents\GitHub\multiwoz-grice-agreement\notebooks\sanity_outputs\fixed_debug_grice_sanity_20260129_104803.jsonl
 - C:\Users\benwe\Documents\GitHub\multiwoz-grice-agreement\notebooks\sanity_outputs\fixed_debug_grice_sanity_20260129_104803.md
Units captured: 29
